<a href="https://colab.research.google.com/github/alfagalileo/Metodos_Computacionales_1/blob/main/_Clase8.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab">
</a>

# Solución de sistemas lineales

## Método iterativo de Jacobi

Este método iterativo se puede resumir en las siguientes etapas.

- *Descomposición.* La matriz de coeficientes $\mathbb{A}$ se descompone en su parte diagonal $\mathbb{D}$ y el resto $\mathbb{R}$, de modo que $\mathbb{A} = \mathbb{D} + \mathbb{R}$.

- *Fórmula de iteración.* Usando esta descomposición, el sistema se reescribe como

$$
\begin{align}
  \mathbb{A} \vec{x} & = \vec{b}\\
  (\mathbb{D}+\mathbb{R}) \vec{x} & = \vec{b}\\
  \mathbb{D} \vec{x} & = \vec{b} -\mathbb{R} \vec{x}\\
  \mathbb{D}^{-1} \mathbb{D} \vec{x} & = \mathbb{D}^{-1} (\vec{b} -\mathbb{R} \vec{x})\\
  \vec{x} & = \mathbb{D}^{-1} (\vec{b} -\mathbb{R} \vec{x})
\end{align}
$$

lo que sugiere el esquema iterativo

$$
  \vec{x}^{(k+1)} = \mathbb{D}^{-1} \big(\vec{b} -\mathbb{R}\, \vec{x}^{(k)}\big).
$$

Aquí, $\mathbb{D}^{-1}$ representa la inversa de la matriz diagonal y $\vec{x}^{(k)}$ es la $k$-ésima aproximación de la solución del sistema. _Nota._ La inversa de una matriz diagonal es otra matriz diagonal cuyos elementos son los recíprocos de los elementos originales, $\mathbb{D}^{-1} = \operatorname{diag}\left(1/a_{11}, 1/a_{22}, \dots, 1/a_{nn}\right)$, por lo que se requiere $a_{ii} \neq 0$.

- *Ansatz inicial.* El método requiere un _ansatz_ de solución inicial $\vec{x}^{(0)}$, y se itera hasta que la solución converja a una tolerancia deseada.

- *Criterio de parada.* Se mide la distancia entre soluciones sucesivas y se detiene la iteración si esta distancia es menor que un valor pequeño de referencia $\epsilon$, es decir,

$$
    \big\|\vec{x}^{(k+1)} - \vec{x}^{(k)}\big\| < \epsilon .
$$

## Condiciones de convergencia

- *Dominancia diagonal.* Una condición suficiente para la convergencia es que la matriz de coeficientes $\mathbb{A}$ sea estrictamente diagonalmente dominante, es decir, que en cada fila la magnitud del elemento diagonal sea mayor que la suma de las magnitudes de los demás elementos de esa fila,

$$
    |a_{ii}| > \sum_{j \neq i} |a_{ij}|, \qquad i = 1, \dots, n .
$$

Esta condición no es necesaria; existen matrices que no la cumplen y para las cuales el método converge.

- *Radio espectral.* La condición necesaria y suficiente se expresa con el radio espectral de la matriz de iteración $\mathbb{T}_J = -\mathbb{D}^{-1}\mathbb{R}$. El método converge para cualquier $\vec{x}^{(0)}$ si y solo si $\rho(\mathbb{T}_J) = \max_i |\lambda_i| < 1$, lo que equivale a

$$
\det\big(\lambda \mathbb{D} + \mathbb{R}\big) = 0
\quad \Longrightarrow \quad
|\lambda| < 1
\qquad \forall\, \lambda \in \mathbb{C} .
$$

In [2]:
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

# Ecuación normal de regresión

Dado un conjunto de datos $\{(x^{(i)}, y^{(i)})\}_{i=1}^{m}$, buscamos una función $h(x) = \theta_1 f_1(x) + \theta_2 f_2(x) + \cdots + \theta_n f_n(x)$ que represente de mejor manera la relación entre las variables.

$$
    h(x) = \begin{pmatrix}
        \theta_1 & \theta_2 & \cdots & \theta_n
    \end{pmatrix}
    \cdot
    \begin{pmatrix}
        f_1(x) \\
        f_2(x) \\
        \vdots \\
        f_n(x)
    \end{pmatrix}
    = \vec{\theta}^T \cdot \vec{F}(x)
$$

Para ello necesitamos minimizar la función de costo

$$
    J(\vec{\theta}) = \frac{1}{2m} \sum_{i=1}^{m} \left(\vec{\theta}^T \vec{F}(x^{(i)}) - y^{(i)}\right)^2
$$

## Forma matricial

Se construye la matriz de diseño $\Phi \in \mathbb{R}^{m \times n}$, cuya fila $i$ es $\vec{F}(x^{(i)})^T$, y el vector de observaciones $\vec{y}$,

$$
    \Phi = \begin{pmatrix}
        f_1(x^{(1)}) & f_2(x^{(1)}) & \cdots & f_n(x^{(1)}) \\
        f_1(x^{(2)}) & f_2(x^{(2)}) & \cdots & f_n(x^{(2)}) \\
        \vdots & \vdots & \ddots & \vdots \\
        f_1(x^{(m)}) & f_2(x^{(m)}) & \cdots & f_n(x^{(m)})
    \end{pmatrix},
    \qquad
    \vec{y} = \begin{pmatrix} y^{(1)} \\ y^{(2)} \\ \vdots \\ y^{(m)} \end{pmatrix}.
$$

Así la función de costo se escribe como

$$
    J(\vec{\theta}) = \frac{1}{2m}\,\big\|\Phi\vec{\theta} - \vec{y}\big\|^2 .
$$

## Ecuación normal

El mínimo se alcanza donde el gradiente se anula,

$$
    \nabla_{\vec{\theta}} J = \frac{1}{m}\,\Phi^T\big(\Phi\vec{\theta} - \vec{y}\big) = \vec{0},
$$

lo que conduce al sistema lineal de $n \times n$ conocido como **ecuación normal**,

$$
    \boxed{\;\Phi^T \Phi\, \vec{\theta} = \Phi^T \vec{y}\;}
$$

Este sistema tiene la forma $A\vec{\theta} = \vec{b}$, con $A = \Phi^T\Phi$ simétrica y $\vec{b} = \Phi^T\vec{y}$, por lo que puede resolverse con el método de Jacobi.

## Ejemplo. Determinación de la gravedad a partir de un lanzamiento vertical

Se lanza una pelota verticalmente hacia arriba y, con una cámara de alta velocidad, se registra su altura $y$ cada $0.05\ \text{s}$ entre $t = 0$ y $t = 1\ \text{s}$. Las mediciones tienen un error aleatorio de $\sigma = 0.02\ \text{m}$. Para generar los datos, use $y_0 = 1.5\ \text{m}$, $v_0 = 4.0\ \text{m/s}$ y $g = 9.8\ \text{m/s}^2$.

El movimiento se describe con el modelo

$$
    y(t) = \theta_1 + \theta_2\, t + \theta_3\, t^2 ,
$$

que es no lineal en $t$ pero lineal en los parámetros, con funciones base $f_1 = 1$, $f_2 = t$ y $f_3 = t^2$. De la comparación con $y(t) = y_0 + v_0 t - \tfrac{1}{2} g t^2$ se obtiene $g = -2\theta_3$.

In [120]:
fps = 60
t = np.arange(66) / fps 

y = np.array([                   
    1.502, 1.560, 1.632, 1.692, 1.736, 1.792, 1.852, 1.898, 1.946, 1.986, 2.034,
    2.072, 2.104, 2.142, 2.168, 2.190, 2.220, 2.236, 2.264, 2.276, 2.288, 2.296,
    2.314, 2.312, 2.314, 2.314, 2.316, 2.310, 2.302, 2.290, 2.286, 2.256, 2.236,
    2.214, 2.196, 2.172, 2.136, 2.100, 2.064, 2.034, 1.992, 1.948, 1.896, 1.852,
    1.798, 1.744, 1.690, 1.628, 1.568, 1.498, 1.432, 1.362, 1.278, 1.208, 1.128,
    1.046, 0.964, 0.886, 0.784, 0.700, 0.592, 0.500, 0.402, 0.300, 0.196, 0.086,
])

sigma = 0.005            